# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirvik-49/Week-1-FlyRank-AI-Assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector Construction
We build the core tabular matrix by extracting pre-query metrics, scaling continuous fields, handling missing continuous values via median imputation, and one-hot encoding categorical attributes. Target-dependent post-event fields are strictly excluded at build time.

### Feature Vector Construction
We build the core tabular matrix by extracting pre-query metrics, scaling continuous fields, handling missing continuous values via median imputation, and one-hot encoding categorical attributes. Target-dependent post-event fields are strictly excluded at build time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

# 1. Generate clean synthetic dataset simulating query log features
np.random.seed(42)
n_samples = 150

df_raw = pd.DataFrame(
    {
        "query_len": np.random.randint(1, 15, n_samples),
        "doc_age_days": np.random.randint(1, 365, n_samples),
        "hist_ctr": np.random.uniform(0.0, 0.35, n_samples),
        "device_category": np.random.choice(["mobile", "desktop"], n_samples),
        "dwell_time_seconds": np.random.exponential(30, n_samples),  # LEAKAGE FIELD
        "conversion_flag": np.random.choice(
            [0, 1], n_samples, p=[0.85, 0.15]
        ),  # TARGET
    }
)

# 2. Impute missing values & encode categoricals
df_features = df_raw.copy()
df_features["hist_ctr"] = df_features["hist_ctr"].fillna(
    df_features["hist_ctr"].median()
)
df_features = pd.get_dummies(
    df_features, columns=["device_category"], drop_first=True
)

# 3. Form final feature matrix X (excluding post-event fields)
feature_cols = ["query_len", "doc_age_days", "hist_ctr", "device_category_mobile"]
X = df_features[feature_cols]

print(f"Feature matrix built successfully. Matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")

Feature matrix built successfully. Matrix shape: (150, 4)
Features: ['query_len', 'doc_age_days', 'hist_ctr', 'device_category_mobile']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Schema & Lineage Summary
| Feature Name | Type | Missing Value Policy | Point-in-Time Availability |
| :--- | :--- | :--- | :--- |
| `query_len` | Numeric (Int) | None (Default to 0 if null) | Pre-execution (Available at query runtime) |
| `doc_age_days` | Numeric (Int) | Imputed with median age | Pre-execution (Indexed metadata) |
| `hist_ctr` | Numeric (Float) | Median imputation | Pre-execution (Historical batch metric) |
| `device_category` | Categorical | Encoded to binary indicator | Pre-execution (HTTP header metadata) |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify feature types, null presence, and statistical ranges
audit_df = pd.DataFrame(
    {
        "dtype": X.dtypes,
        "null_count": X.isnull().sum(),
        "min": X.min(),
        "max": X.max(),
        "mean": X.mean().round(4),
    }
)

print("--- Feature Audit Report ---")
print(audit_df)

--- Feature Audit Report ---
                          dtype  null_count       min       max      mean
query_len                 int64           0         1        14    7.8933
doc_age_days              int64           0         2       360  191.1733
hist_ctr                float64           0  0.001772  0.346519    0.1802
device_category_mobile     bool           0     False      True    0.5133


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Data Leakage Audit
Features are evaluated against the target variable (`conversion_flag`) to ensure no post-event or target-derived signals exist in the training set. Any feature exhibiting perfect correlation ($|r| > 0.95$) or recorded after the timestamp of prediction is flagged as data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Compute pairwise correlations between all candidate features and the target
target_col = df_features["conversion_flag"]
correlations = df_features.corr(numeric_only=True)["conversion_flag"].drop(
    "conversion_flag"
)

print("--- Correlation Matrix against Target ---")
print(correlations.round(4))

# Automated leakage assertion test
leaked_cols = correlations[correlations.abs() > 0.90].index.tolist()
assert (
    len(leaked_cols) == 0
), f"CRITICAL LEAKAGE DETECTED: High correlation in {leaked_cols}"

print("\nLeakage Test Passed: Zero target-derived features found in matrix X.")

--- Correlation Matrix against Target ---
query_len                 0.0517
doc_age_days              0.0932
hist_ctr                  0.0373
dwell_time_seconds       -0.0397
device_category_mobile   -0.0099
Name: conversion_flag, dtype: float64

Leakage Test Passed: Zero target-derived features found in matrix X.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Features Log
* `dwell_time_seconds`: **Excluded.** Recorded post-click during user session; unavailable at query prediction time.
* `conversion_flag`: **Excluded.** Target label used strictly for ground-truth calculation.
* `user_id` / `client_hash`: **Excluded.** High-cardinality private identifier causing overfitting and privacy risk.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verification script: Ensure prohibited fields never enter the training matrix
prohibited_fields = ["dwell_time_seconds", "conversion_flag", "user_id"]

for field in prohibited_fields:
    assert (
        field not in X.columns
    ), f"Security Error: Prohibited field '{field}' detected in X!"

print(
    f"Verification Complete: All {len(prohibited_fields)} target/privacy fields successfully excluded."
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.